<a href="https://colab.research.google.com/github/satishmathapa/mac247-labs/blob/main/MAC247_LAB1_S01_Reference_Monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 247 Graded Lab 1, Part 1: the reference monitor and the role model

**Part 1, Tuesday, September 22, 2026. Google Colab in a browser. Every cell below uses only the
Python 3 standard library, so there is nothing to install.**

Lab 1 is one investigation in two parts. Today you build the instrument: four access-control
decision functions, tested against a truth table whose expected answers are written down before
the code runs, and then the role model of a small finance system, which says what each role
legitimately needs and which duties must never sit with one person.

On Thursday, September 24, Part 2 (`MAC247_LAB1_S03_Access_Review.ipynb`) builds the roster and
the export a real system would have produced, and runs today's model against it. The model finds
what that system got wrong, and the last cell here tells you in advance what two things are wrong
with it, so you can tell a working detector from one that reports something plausible.

**Nothing in this notebook contacts any host outside it.** Everything is computed here, from
data the notebook writes for itself.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. The token prints into the output, and the
output is what gets saved, so it travels with the notebook into your repository. Use the same
`sid` in Part 2 on Thursday: the assignment card at the end of this notebook is derived from it,
so Part 2 recomputes the card rather than asking you to carry it across.

Before you run anything: if you have not created your repository yet, do it now. Open github.com
in another tab, create a **private** repository named `mac247-labs`, and invite the instructor as
a collaborator under Settings, Collaborators. Everything you produce for this lab goes in a folder
named `lab1`. Do not rush the private setting, because a repository the instructor cannot open is
a lab that cannot be graded.

In [14]:
import hashlib, datetime

sid = '24647180'
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC247 Lab 1 token:', TOKEN)

MAC247 Lab 1 token: f745c1509f3a8662


## Section 2. Four decision functions

A reference monitor answers one question: may this subject perform this operation on this
object, right now. Four models answer it four different ways, and the difference between them
is not syntax, it is who is allowed to change the answer.

Read each function before you run it. In the report you are asked which model fits which
organization, and you cannot answer that from the output alone.

### DAC, discretionary access control

The owner of an object decides. The decision is a lookup in a list the owner controls, which is
exactly its strength for two people and exactly its weakness for two thousand.

In [16]:
# object -> {subject -> permission string, using r, w and - in fixed positions}
ACL = {
    'ledger.csv':   {'owner': 'finance_lead', 'analyst_a': 'r--', 'finance_lead': 'rw-'},
    'payroll.db':   {'owner': 'finance_lead', 'finance_lead': 'rw-'},
    'backup.tar':   {'owner': 'svc_backup',   'svc_backup': 'rw-', 'auditor_a': 'r--'},
    'audit.log':    {'owner': 'auditor_a',    'auditor_a': 'r--'},
}
BIT = {'read': 0, 'write': 1}

def dac(subject, obj, op):
    """True when the object's own access list grants the subject that operation."""
    perms = ACL.get(obj, {}).get(subject, '---')
    return perms[BIT[op]] != '-'

### MAC, mandatory access control

A central authority sets a clearance on every subject and a classification on every object, and
no owner can override either. The Bell-LaPadula confidentiality rules are two lines: a subject
may read at or below its clearance, and may write at or above it. The second rule is the one
students get wrong. Writing *down* is forbidden because that is how classified content leaks
into an unclassified file, whether or not anybody meant it to.

In [17]:
LEVEL = {'public': 0, 'internal': 1, 'confidential': 2, 'restricted': 3}
CLEARANCE = {'finance_lead': 'restricted', 'analyst_a': 'internal',
             'auditor_a': 'confidential', 'svc_backup': 'restricted'}
CLASSIFICATION = {'ledger.csv': 'confidential', 'payroll.db': 'restricted',
                  'backup.tar': 'restricted', 'audit.log': 'internal'}

def mac(subject, obj, op):
    """Bell-LaPadula: no read up, no write down."""
    s = LEVEL[CLEARANCE.get(subject, 'public')]
    o = LEVEL[CLASSIFICATION.get(obj, 'public')]
    return s >= o if op == 'read' else s <= o

### RBAC, role-based access control

Permissions attach to roles and people attach to roles, so the thing an administrator edits when
somebody changes job is one membership rather than every access list that person appears in.
This is the model almost every organization above a certain size actually runs.

In [18]:
ROLE_PERMS = {
    'finance':  {'read': {'ledger.csv', 'payroll.db'}, 'write': {'ledger.csv', 'payroll.db'}},
    'analyst':  {'read': {'ledger.csv'},               'write': set()},
    'auditor':  {'read': {'ledger.csv', 'audit.log', 'backup.tar'}, 'write': set()},
    'backup':   {'read': {'*'},                        'write': {'backup.tar'}},
}
USER_ROLE = {'finance_lead': 'finance', 'analyst_a': 'analyst',
             'auditor_a': 'auditor', 'svc_backup': 'backup'}

def rbac(subject, obj, op):
    perms = ROLE_PERMS.get(USER_ROLE.get(subject, ''), {})
    allowed = perms.get(op, set())
    return obj in allowed or '*' in allowed

### ABAC, attribute-based access control

The decision is an expression over attributes of the subject, the object and the environment, so
it can say things the other three cannot say at all: not at 2 AM, not from an unmanaged laptop,
not outside your own department. The cost is that the policy is now a program, and a program can
be wrong in ways a list cannot.

In [19]:
def abac(subject_attrs, object_attrs, env):
    """Every clause must hold. Returns (decision, the first clause that failed)."""
    clauses = [
        ('department matches',   subject_attrs['department'] == object_attrs['department']),
        ('clearance sufficient', LEVEL[subject_attrs['clearance']] >= LEVEL[object_attrs['sensitivity']]),
        ('within work hours',    9 <= env['hour'] <= 18),
        ('managed device',       env['managed_device'] is True),
    ]
    for name, ok in clauses:
        if not ok:
            return False, name
    return True, None

### The truth table

Six cases with an expected answer written down before the code ran. If a line reads FAIL, the
model in your head and the model in the function disagree, and the thing to fix is the function
or your reading of the model. Do not edit the expected value to make the line go green.

In [20]:
CASES = [
    ('DAC  analyst_a writes ledger.csv',     dac('analyst_a', 'ledger.csv', 'write'),   False),
    ('DAC  auditor_a reads backup.tar',      dac('auditor_a', 'backup.tar', 'read'),    True),
    ('MAC  analyst_a reads ledger.csv',      mac('analyst_a', 'ledger.csv', 'read'),    False),
    ('MAC  finance_lead writes audit.log',   mac('finance_lead', 'audit.log', 'write'), False),
    ('RBAC auditor_a reads audit.log',       rbac('auditor_a', 'audit.log', 'read'),    True),
    ('RBAC analyst_a writes payroll.db',     rbac('analyst_a', 'payroll.db', 'write'),  False),
]
bad = 0
for name, got, want in CASES:
    ok = (got == want)
    bad += (not ok)
    print(f'{"PASS" if ok else "FAIL"}  {name:36s} decision={str(got):5s} expected={want}')
assert bad == 0, 'a decision function disagrees with the model it claims to implement'
print('\n  CHECK PASS: all six decisions match the model')
print('  token:', TOKEN)

PASS  DAC  analyst_a writes ledger.csv     decision=False expected=False
PASS  DAC  auditor_a reads backup.tar      decision=True  expected=True
PASS  MAC  analyst_a reads ledger.csv      decision=False expected=False
PASS  MAC  finance_lead writes audit.log   decision=False expected=False
PASS  RBAC auditor_a reads audit.log       decision=True  expected=True
PASS  RBAC analyst_a writes payroll.db     decision=False expected=False

  CHECK PASS: all six decisions match the model
  token: f745c1509f3a8662


### ABAC, the same request at four different moments

The same subject, the same object, four environments. This is the property the other three
models cannot express, and it is worth seeing rather than reading about.

In [22]:
subj = {'department': 'finance', 'clearance': 'confidential'}
obj  = {'department': 'finance', 'sensitivity': 'confidential'}
ENVS = [
    ('Tuesday 10:00, college laptop',   {'hour': 10, 'managed_device': True},  True),
    ('Tuesday 02:00, college laptop',   {'hour': 2,  'managed_device': True},  False),
    ('Tuesday 10:00, personal laptop',  {'hour': 10, 'managed_device': False}, False),
    ('Tuesday 19:00, college laptop',   {'hour': 19, 'managed_device': True},  False),
]
for label, env, want in ENVS:
    decision, failed = abac(subj, obj, env)
    assert decision == want, f'ABAC disagreed on: {label}'
    print(f'{"ALLOW" if decision else "DENY "}  {label:34s} {"" if decision else "blocked by: " + failed}')
print('\n  CHECK PASS: ABAC denied three of four identical requests on environment alone')

ALLOW  Tuesday 10:00, college laptop      
DENY   Tuesday 02:00, college laptop      blocked by: within work hours
DENY   Tuesday 10:00, personal laptop     blocked by: managed device
DENY   Tuesday 19:00, college laptop      blocked by: within work hours

  CHECK PASS: ABAC denied three of four identical requests on environment alone


## Section 3. The role model

This is the model of the finance system as it is supposed to be: four roles, four accounts, the
entitlements each role legitimately needs, and the pairs of duties that must never sit with one
person. Part 2 builds a roster from exactly this and judges it.

`ROLE_REQUIRED` is a least-privilege statement. An account holding an entitlement that is not
in its role's set is over-entitled, whether or not anybody has abused it yet.

`SOD_CONFLICTS` is a separation-of-duties statement. Each pair names two capabilities that are
individually reasonable and jointly dangerous, because together they let one person both do a
thing and erase the record of having done it. A model is allowed to name a conflict that today's
population does not contain; that is what makes it a standard rather than a description.

In [32]:
ROLE_REQUIRED = {
    'finance':  {'group:mac247fin'},
    'analyst':  {'group:mac247ana'},
    'auditor':  {'group:mac247aud'},
    'backup':   {'group:mac247bkp'},
}
ROSTER = {  # the account each role gets on Thursday
    'finance':  'mac247_finlead',
    'analyst':  'mac247_analyst',
    'auditor':  'mac247_auditor',
    'backup':   'mac247_backup',
}
SOD_CONFLICTS = [
    ({'group:mac247fin'}, {'group:mac247aud'},
     'posting a transaction and signing off the audit of it'),
    ({'group:mac247bkp'}, {'group:mac247aud'},
     'holding the backups and attesting that the backups are intact'),
    ({'sudo:ALL'},        {'group:mac247aud'},
     'unrestricted administration and custody of the audit record'),
]
print('roles    :', ', '.join(sorted(ROLE_REQUIRED)))
print('accounts :', ', '.join(ROSTER[r] for r in sorted(ROSTER)))
print('conflicts:', len(SOD_CONFLICTS))

roles    : analyst, auditor, backup, finance
accounts : mac247_analyst, mac247_auditor, mac247_backup, mac247_finlead
conflicts: 3


### Your assignment card

Two faults are planted in the roster Part 2 reviews, and which two is decided by your student
ID. Run the cell and write the card down, or keep this notebook saved: Part 2 recomputes the
same card from the same `sid`, and the instructor can recompute what yours should have been.

The point of planting them deliberately is that you know the ground truth before the detector
runs. When Part 2 reports violations you can check whether it found the right ones, which is
the only way to tell a working detector from one that reports something plausible.

The extra entitlement is never one the role already requires, and it always collides with a
duty that role already holds, so it is both an over-entitlement and a separation-of-duties
break. The cell asserts both of those before it prints anything.

In [30]:
h = int(hashlib.sha256((sid + '-lab1-card').encode()).hexdigest(), 16)

EXCESS_POOL = ['group:mac247fin', 'group:mac247aud', 'group:mac247bkp', 'sudo:ALL']

def conflicting_additions(role):
    """Entitlements that the role does not require and that collide with one it holds."""
    held = ROLE_REQUIRED[role]
    out = []
    for e in EXCESS_POOL:
        if e in held:
            continue
        for a, b, _ in SOD_CONFLICTS:
            if (a <= held and b <= {e}) or (b <= held and a <= {e}):
                out.append(e)
                break
    return out

roles = sorted(ROLE_REQUIRED)
candidates = [r for r in roles if conflicting_additions(r)]
excess_role = candidates[h % len(candidates)]
choices = conflicting_additions(excess_role)
excess_entitlement = choices[(h // 7) % len(choices)]
stale_role = roles[(h // 31) % len(roles)]

CARD = {
    'fault_1_over_entitled_account': ROSTER[excess_role],
    'fault_1_extra_entitlement': excess_entitlement,
    'fault_2_stale_account': ROSTER[stale_role],
}
for k, v in CARD.items():
    print(f'{k:32s} {v}')

assert excess_entitlement not in ROLE_REQUIRED[excess_role], 'not actually an excess'
assert excess_entitlement in conflicting_additions(excess_role), 'not actually a conflict'
print('\n  CHECK PASS: the extra entitlement is outside the role that receives it and '
      'collides with a duty it already holds')
print('  token:', TOKEN)

fault_1_over_entitled_account    mac247_auditor
fault_1_extra_entitlement        group:mac247fin
fault_2_stale_account            mac247_backup

  CHECK PASS: the extra entitlement is outside the role that receives it and collides with a duty it already holds
  token: f745c1509f3a8662


## Closing Part 1

Three `CHECK PASS` lines and one block of six `PASS` lines must appear above. Then, about 20
minutes before class ends:

1. **File, then Save a copy in GitHub.** Choose `mac247-labs`, set the path to
   `lab1/MAC247_LAB1_S01_Reference_Monitor.ipynb`, and use `lab1 part 1` as the commit message.
   The first time, Colab asks to connect to GitHub; allow it, so your private repository appears.
2. Open the commit on github.com and confirm the notebook is actually there, with your token
   visible in the saved output. A save that failed silently looks exactly like one that worked
   until you look.
3. Screenshots, both showing your Google account name and your token: `lab1_s01_repo_created.png`,
   the Settings, Collaborators page of your private repository with the instructor's invitation
   listed; and `lab1_s01_model_checks.png`, this notebook showing the six `PASS` lines, the
   assignment card and your token. Upload both to `lab1` on github.com with Add file, then
   Upload files.
4. Sign out: Runtime, then Disconnect and delete runtime, and on a shared computer also sign out
   of Google and GitHub in the browser. Capture `lab1_s01_logout.png` for your report.

## What goes in the report

1. Which of the four models fits a two-person startup, which fits a fifty-thousand-person
   defense contractor, and which fits an internal data lake where the rule depends on the row.
   One sentence each, and the sentence has to name the property that decides it, not the model.
2. MAC denied `finance_lead` a write to `audit.log` even though `finance_lead` holds the highest
   clearance in the table. Explain the denial. Then say what it protects against, in terms of a
   thing that could otherwise end up in that file.
3. ABAC denied three of four identical requests. Name the clause that did the work in each, and
   say which of the other three models could have expressed that clause.
4. Your assignment card names two faults. For each, write one sentence predicting what Part 2's
   analysis should report, before you have seen any of it.